In [ ]:
!pip install pyspark

In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName("Covid Data Mining").getOrCreate()
df = spark.read.csv("/kaggle/input/datasets/caesarmario/our-world-in-data-covid19-dataset/owid-covid-data.csv", header=True, inferSchema=True)

In [ ]:
df.show()

In [ ]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

df.printSchema()

df.show(5)

In [ ]:
from pyspark.sql.functions import col, sum, round

total_rows = df.count()

null_percentage = df.select([
    round(
        sum(col(c).isNull().cast("int")) / total_rows * 100,
        2
    ).alias(c)
    for c in df.columns
])

null_percentage.show()

In [ ]:
from pyspark.sql.functions import col

numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.dataType.typeName() in ["double", "integer", "long", "float"]
]

correlations = []

for c in numeric_cols:
    if c != "new_cases":
        corr_value = df.stat.corr(c, "new_cases")
        correlations.append((c, corr_value))

corr_df = spark.createDataFrame(
    correlations,
    ["column", "correlation"]
)

corr_df.orderBy(
    col("correlation").desc()
).show(50)

In [ ]:
from pyspark.sql.functions import col, sum, round


total_rows = df.count()


numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.dataType.typeName() in ["double", "integer", "long", "float"]
]


results = []

for c in numeric_cols:

    null_count = df.filter(col(c).isNull()).count()

    null_percentage = (null_count / total_rows) * 100
    correlation = df.stat.corr(c, "new_cases")

    data_type = df.schema[c].dataType.typeName()

    results.append(
        (
            c,
            data_type,
            null_count,
            null_percentage,
            correlation
        )
    )


analysis_df = spark.createDataFrame(
    results,
    [
        "column",
        "data_type",
        "null_count",
        "null_percentage",
        "correlation_with_new_cases"
    ]
)


analysis_df.orderBy(
    col("correlation_with_new_cases").desc()
).show(100)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


data = df.filter(F.col("continent").isNotNull())


data = data.filter(F.col("new_cases").isNotNull())


vacc_cols = [
    "people_vaccinated_per_hundred",
    "people_fully_vaccinated_per_hundred",
    "total_vaccinations_per_hundred",
]
data = data.fillna(0, subset=vacc_cols)

print("Rows:", data.count(), "| Countries:", data.select("location").distinct().count())

In [ ]:
w = Window.partitionBy("location").orderBy("date")
w7 = w.rowsBetween(-7, -1)     # previous 7 days, excluding today
w14 = w.rowsBetween(-14, -1)

data = (
    data
    .withColumn("cases_lag_1", F.lag("new_cases", 1).over(w))
    .withColumn("cases_lag_7", F.lag("new_cases", 7).over(w))
    .withColumn("cases_lag_14", F.lag("new_cases", 14).over(w))
    .withColumn("cases_roll_7", F.avg("new_cases").over(w7))
    .withColumn("cases_roll_14", F.avg("new_cases").over(w14))
    .withColumn("deaths_roll_7", F.avg("new_deaths").over(w7))
    .withColumn("day_of_week", F.dayofweek("date").cast("double"))
    .withColumn("month", F.month("date").cast("double"))
)


data = data.dropna(subset=["cases_lag_1", "cases_lag_7", "cases_roll_7"])

data.select("location", "date", "new_cases", "cases_lag_1", "cases_lag_7", "cases_roll_7").show(5)

In [ ]:
dates = [r[0] for r in data.select("date").distinct().orderBy("date").collect()]
cutoff = dates[int(len(dates) * 0.8)]
print("Train: before", cutoff, "| Test: from", cutoff)

train = data.filter(F.col("date") < cutoff).cache()
test = data.filter(F.col("date") >= cutoff).cache()
print("Train rows:", train.count(), "| Test rows:", test.count())

In [ ]:
static_cols = [
    "population", "population_density", "median_age", "aged_65_older", "aged_70_older",
    "gdp_per_capita", "cardiovasc_death_rate", "diabetes_prevalence",
    "hospital_beds_per_thousand", "life_expectancy", "human_development_index",
]
dynamic_cols = ["stringency_index"] + vacc_cols
history_cols = [
    "cases_lag_1", "cases_lag_7", "cases_lag_14",
    "cases_roll_7", "cases_roll_14", "deaths_roll_7",
    "day_of_week", "month",
]
candidate_cols = static_cols + dynamic_cols + history_cols


n_train = train.count()
null_pct = (
    train.select([(F.sum(F.col(c).isNull().cast("int")) / n_train * 100).alias(c) for c in candidate_cols])
    .first()
    .asDict()
)

feature_cols = [c for c in candidate_cols if null_pct[c] < 60]
dropped = [c for c in candidate_cols if null_pct[c] >= 60]
print("Using", len(feature_cols), "features")
print("Dropped (>=60% null):", dropped)

In [ ]:
from pyspark.ml.feature import Imputer, StringIndexer, VectorAssembler

imputed_cols = [c + "_imp" for c in feature_cols]

imputer = Imputer(
    inputCols=feature_cols,
    outputCols=imputed_cols,
    strategy="median"
)

cont_idx = StringIndexer(
    inputCol="continent",
    outputCol="continent_idx",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=imputed_cols + ["continent_idx"],
    outputCol="features"
)

prep_stages = [imputer, cont_idx, assembler]

In [ ]:
# from pyspark.ml.regression import DecisionTreeRegressor
# from pyspark.ml import Pipeline

# dt = DecisionTreeRegressor(
#     featuresCol="features",
#     labelCol="new_cases",
#     maxDepth=10,
#     seed=42
# )

# pipeline_dt = Pipeline(
#     stages=prep_stages + [dt]
# )

In [ ]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="new_cases",
    numTrees=20,
    maxDepth=4,
    seed=42
)

pipeline_rf = Pipeline(
    stages=prep_stages + [rf]
)

df_model = pipeline_rf.fit(train)

In [ ]:
dt_predictions = df_model.transform(test)

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator_mae = RegressionEvaluator(
    labelCol="new_cases",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_rmse = RegressionEvaluator(
    labelCol="new_cases",
    predictionCol="prediction",
    metricName="rmse"
)


mae = evaluator_mae.evaluate(dt_predictions)
rmse = evaluator_rmse.evaluate(dt_predictions)


print("Decision Tree")
print("MAE :", mae)
